# Part 1


## Introduction

An online resource was consulted for the purposes of rendering gymnasium environments in jupyter notebooks. This resource can be found at the following link and is referenced multiple times throughout this assignment:

- Source A: https://community.latenode.com/t/how-to-render-gymnasium-environment-inside-jupyter-without-external-window/30582/4

In [ ]:
# =============================== Taken from Source A ===================
import sys
from pathlib import Path

for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "src" / "rl_suite").is_dir():
        sys.path.insert(0, str(_p / "src"))
        break
# =======================================================================
import gymnasium as gym
import pprint  # useful for printing nested items

import rl_suite as rl

utils = rl.RLToolbox.utils


In [ ]:
ENV_ID, SEED = "CartPole-v1", 10
NUM_TIMESTEPS_GOAL = 10000
env = gym.make(ENV_ID, max_episode_steps=NUM_TIMESTEPS_GOAL)
FOUR_DIM_DISCRETIZATION = {
    "num_bins": 10,
    "intervals": ((-2.4, 2.4), (-2.5, 2.5), (-0.2095, 0.2095), (-3.5, 3.5)),
}

TWO_DIM_DISCRETIZATION = {
    "num_bins": 100,
    "feature_indices": (0, 2),
    "intervals": ((-2.4, 2.4), (-0.2095, 0.2095)),
}


In [ ]:
# ``print_discrete_space`` is accessed through ``rl_suite.RLToolbox.utils``.


## Approach

It is clear that the duration of this control algorithm depends directly on how we discretize the continuous feedback; so our execution loop will be a function of the fineness of discretization. 

The algorithm for off-policy MC control algorithm involves using an infinite amount of episodes but this is obviously infeasible. We need to generate enough episodes that the agent can actually solve/optimize the problem (the target policy must converge to a deterministic optimum). The instructions say the goal is for the solver to balance it for 10,000 steps. Then our goal is to keep generating episodes until we finally reach an episode that is 10,000 timesteps long, however this is impractical so we implemented a maximum number of iterations of 100,000. Since the behaviour policy used to generate the episodes is arbitrarily soft and independent of the target policy being learned (except for coverage), we cannot use these episodes to test whether our algorithm has converged. So for every 1000 iterations, we generate an episode using the target policy as a test. Therefore, the agent has 100 "test attempts" to balance the cart for 10,000 timesteps, and 100,000 episodes to learn from. If the agent still does not accomplish this, it has failed.

We implemented three modified versions of the Off-Policy Control algorithm; each agent failed. We will present each implementation, discuss the changes, and afterward discuss the limitations of this approach. The "logs" for these algorithms can be found in Appendices B, C, and D.

Here is the main loop used to execute the environment (generate an episode) for agents 1 and 2:

In [ ]:
# Environment execution and discretization are handled by ``rl_suite.RLToolbox.utils.RLEnvironmentRunner``.


The shared off-policy Monte Carlo implementation lives in `rl_suite.algorithms`; this notebook varies the discretized environment interface and behaviour policy instead of defining separate agent classes inline.

In [ ]:
# Off-policy MC control is available as ``agent.mc.off_policy``.


## Experiment 1

This first experiment uses `OffPolicyMCAgent` with a four-dimensional CartPole discretization. The observed feedback includes cart position, cart velocity, angle (in radians), and angular velocity. Since it is infeasible to have an infinite state space for MC control, we manually limit the range of the cart velocity and the angular velocity. We chose the smallest range that included every single observed value from initial testing; these were $(-2.5,2.5)$ and $(-3.5,3.5)$ for cart velocity and angular velocity, respectively.

As mentioned earlier, discretization is necessary, and we decided to settle on 10 "bins" for each variable (so each variable has 10 possible values). Thus, the size of our state space is $|S| = 10^4 = 10000$, and since there are only two possible actions (push left or push right), our state-action is of size $|S \times A| = 10000 \times 2 = 20000$. For each state variables we ensure that there is an equal number of bins representing negative values as positive values (the edge case of 0 is irrelevant). Thus the number of bins are forced to be even.

The behaviour policy simply chooses between action 0 and 1 with equal probability for any state. Thus it is a soft policy and the assumption of coverage is still valid (any state-action pair possible under the target policy is possible under behaviour).

In [ ]:
# Experiment 1: four-dimensional CartPole discretization with a uniform behaviour policy.


In [ ]:
agent = rl.algorithms(env)
four_dim_runner = utils.RLEnvironmentRunner(env, discretization=FOUR_DIM_DISCRETIZATION)
utils.print_discrete_space(four_dim_runner.discrete_space)
four_dim_mc = agent.mc.off_policy
four_dim_mc.gamma = 0.9
four_dim_mc.behavior = "uniform"
four_dim_output = four_dim_mc.control(
    four_dim_runner,
    num_timesteps_goal=NUM_TIMESTEPS_GOAL,
    close_env=False,
)


## Experiment 2

The first experiment failed to converge to an optimal target policy. We realized that the range of starting values enforced by the environment is very small (from -0.05 to 0.05). This meant that for each of our variables, the starting value could only belong to one of the bins. While this is fine theoretically (we do not need the Exploring Starts assumption for Off-Policy MC), it menas that very few of the states are being sampled frequently to make learning practical (theoretically we need to be able to guarantee each state-action pair is visited infinitely). Given our current discretization this would take an infeasibly long time unless we significantly increase the number of bins (which quickly becomes computationally intractable without distributed architecture).

We then remembered that there are only two possible actions: to push left or right. There is no way for the agent to decide how hard to push at any given timestep; it can only apply a pre-determined constant amount of pressure regardless of any state variables. The only thing the agent needs to decide is the direction to move the cart to. Therefore, the cart velocity and angular velocity are practically useless information! The only pertinent information for deciding on a direction to push are the position of the cart and angle of the pole (regardless of velocity).

We will now only consider cart position and pole angle and use $100$ bins for each. Therefore the size of our state space $|S| = 100 \times 100 = 10000$ remains the same. By eliminating the useless velocity variables, we have gained a drastically finer discretization without an increase in the size of the state space!

In [ ]:
# Experiment 2: two-dimensional CartPole discretization with the same MC algorithm.


In [ ]:
two_dim_runner = utils.RLEnvironmentRunner(env, discretization=TWO_DIM_DISCRETIZATION)
utils.print_discrete_space(two_dim_runner.discrete_space)
two_dim_agent = rl.algorithms(env)
two_dim_mc = two_dim_agent.mc.off_policy
two_dim_mc.gamma = 0.9
two_dim_mc.behavior = "uniform"
two_dim_output = two_dim_mc.control(
    two_dim_runner,
    num_timesteps_goal=NUM_TIMESTEPS_GOAL,
    close_env=False,
)


## Experiment 3

The second experiment also failed to converge despite our significantly finer discretization. We came up with the hypothesis that there is too much noise in our behaviour policy; by choosing either action with equal probability this policy is ignoring new information about the state-action pairs which is used to update the target policy. So then our final idea is to bring the behaviour policy closer to the target policy whilst retaining its "softness". This is done by providing a small epsilon term to the algorithm. At any point in an episode, the behaviour policy will choose the current best action (greedy argmax from target policy) with probability $1-\epsilon$, and with probability $\epsilon$ it will pick the non-greedy action for exploration purposes. Thus our behaviour policy is now an $\epsilon$-soft policy, but it still maintains coverage of the target policy.

To do this, however, we need to keep track of the probability for each action chosen by the behaviour policy. This was not needed before because both actions had an equal probability so we could simply use a hard code value of $0.5$. Clearly, $\epsilon \neq 1 - \epsilon \neq 0.5$, so the MC agent records the behaviour-policy probability with each generated transition. We also decided to decrease our gamma value (thereby decreasing the long term return and ultimately punishing the agent for shorter episodes).

In [ ]:
# Experiment 3 reuses the two-dimensional runner and changes only the behaviour policy.


This experiment reuses the two-dimensional discretization and changes only the behaviour policy configured on `OffPolicyMCAgent`.

In [ ]:
# Epsilon-soft behaviour is configured on the package MC control agent.


In [ ]:
epsilon_soft_runner = utils.RLEnvironmentRunner(env, discretization=TWO_DIM_DISCRETIZATION)
utils.print_discrete_space(epsilon_soft_runner.discrete_space)
epsilon_soft_agent = rl.algorithms(env)
epsilon_soft_mc = epsilon_soft_agent.mc.off_policy
epsilon_soft_mc.gamma = 0.3
epsilon_soft_mc.behavior = "epsilon_soft"
epsilon_soft_mc.epsilon = 0.1
epsilon_soft_output = epsilon_soft_mc.control(
    epsilon_soft_runner,
    num_timesteps_goal=NUM_TIMESTEPS_GOAL,
    close_env=False,
)


## Limitations

So why did our algorithms fail to converge to a deterministic optimal policy? Perhaps an even finer discretization (larger state space) is needed. perhaps a more selective discretization that involves non-linear transformations of the current intervals are needed (to emphasize states that are less likely to be encountered).

It is also important to note that this is a very limited environment; as mentioned earlier, the only actions the agent can take is to decide the direction in which to push the cart. The agent cannot specify the amount of force to apply at any timestep, nor can it decide not to interfere. It seems nearly impossible that a constant force applied at each timestep could ever enable control of this problem. As the documentation notes, the "center of gravity of the pole varies the amount of energy needed to move the cart underneath it". Perhaps this problem with the given state and action space could be better solved by non-tabular RL methods. Our conclusion is that for any commercial PC and GPU, solving this given problem (keeping the pole balanced indefinitely) with this algorithm and state-action space, is not possible.

# Appendix A

Here are the logs from the four-dimensional discretization target policy tests.

In [ ]:
pprint.pprint(four_dim_output)


# Appendix B

Here are the logs from the two-dimensional discretization target policy tests:

In [ ]:
pprint.pprint(two_dim_output)


# Appendix C

Here are the logs from the epsilon-soft behaviour-policy target policy tests:

In [ ]:
pprint.pprint(epsilon_soft_output)
